# 🔬 Prototipação de ML (Séries Temporais) - Clima × Dengue
Neste laboratório, substituímos o HistGradientBoostingRegressor pelo **Prophet** (IA especializada em Séries Temporais). 
Nosso objetivo é carregar os dados classificados por latitude, isolar um município para treinar o modelo e avaliar a acurácia.

## 1. Configuração e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from prophet import Prophet

plt.style.use('ggplot')

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

## 2. Carregamento da Base Completa (Painel de Latitudes)
Os dados brutos já foram processados pelo pipeline e divididos em 8 faixas de latitude (de 22.0°S a 34.0°S). Vamos carregar o painel final.

In [2]:
from src.processamento.motor_lote_modelagem_v2_por_latitude import build_base_panel

print("Carregando painel completo de dados...")
painel, lookup = build_base_panel(BASE_DIR)
print(f"Total de registros: {len(painel):,}")

16:29:00 [INFO] build_base_panel → fonte canônica: output/csvs_latitude/ (1191 municípios esperados)


Carregando painel completo de dados...
Total de registros: 142,800


## 3. Isolando um Município (Experimentação Ágil)

In [3]:
cidade_alvo = 'joinville'
df_cidade = painel[painel['city_name'].str.lower().str.contains(cidade_alvo, na=False)].copy()

if df_cidade.empty:
    codigo_maior_pico = painel.groupby('code')['dengue_cases'].sum().idxmax()
    df_cidade = painel[painel['code'] == codigo_maior_pico].copy()

df_cidade['data'] = pd.to_datetime(
    df_cidade['year'].astype(str) + '-' + df_cidade['month'].astype(str).str.zfill(2)
)
df_cidade = df_cidade.sort_values('data').reset_index(drop=True)
nome_cidade = df_cidade['city_name'].iloc[0].upper()
print(f"{nome_cidade} isolada para análise.")

JOINVILLE isolada para análise.


## 4. Feature Engineering (Lags)

In [4]:
# Memória climática (3 meses de atraso)
df_cidade['temp_lag3'] = df_cidade['temp_media'].shift(3)
df_cidade['prec_lag3'] = df_cidade['precipitacao'].shift(3)
df_cidade['enso_lag3'] = df_cidade['enso'].shift(3)

df_ml = df_cidade.dropna(subset=['temp_lag3', 'prec_lag3', 'enso_lag3']).copy()

## 5. Treinamento do Modelo Avançado (Prophet)
O Prophet exige duas colunas específicas: ds (Data) e y (Casos Reais). Ele foi feito exatamente para entender sazonalidade climática (inverno/verão).

In [5]:
from sklearn.metrics import mean_absolute_error, r2_score

# Preparando os dados no padrão do Prophet
df_prophet = df_ml.copy()
df_prophet = df_prophet.rename(columns={'data': 'ds', 'dengue_cases': 'y'})

# Separando Treino (Histórico até 2022) e Teste (O futuro: 2023-2024)
mask_treino = df_prophet['ds'].dt.year <= 2022
mask_teste = df_prophet['ds'].dt.year > 2022

treino = df_prophet[mask_treino]
teste = df_prophet[mask_teste]

# Configurando a IA de Séries Temporais (Sazonalidade anual ativada)
modelo_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)

# Adicionando o Clima como causas extras (Regressores)
modelo_prophet.add_regressor('temp_lag3')
modelo_prophet.add_regressor('prec_lag3')
modelo_prophet.add_regressor('enso_lag3')

# Treinando o modelo matematicamente
modelo_prophet.fit(treino)

# Pedindo para ele prever os casos de Dengue no período de Teste
futuro = teste[['ds', 'temp_lag3', 'prec_lag3', 'enso_lag3']].copy()
previsoes = modelo_prophet.predict(futuro)

# Limpeza: A IA pode prever -5 casos, mas isso não existe. Zeramos os negativos.
predicoes = np.clip(previsoes['yhat'].values, 0, None)
y_teste = teste['y'].values

print("Treinamento Prophet concluído!")
print(f"MAE (Erro Médio Absoluto): {mean_absolute_error(y_teste, predicoes):.1f} casos/mês")
print(f"R² Score: {r2_score(y_teste, predicoes):.2f}")

15:35:50 [INFO] Chain [1] start processing
15:35:50 [INFO] Chain [1] done processing


Treinamento Prophet concluído!
MAE (Erro Médio Absoluto): 0.3 casos/mês
R² Score: 0.05


## 6. Verificação Visual (Realidade x Prophet)

In [6]:
plt.figure(figsize=(14, 6))

# Linha Real
plt.plot(teste['ds'], y_teste, label='Dengue REAL', color='black', linewidth=2, marker='o')

# Linha Prevista pelo Prophet
plt.plot(teste['ds'], predicoes, label='Previsão PROPHET', color='dodgerblue', linewidth=2, linestyle='dashed', marker='x')

# Área de incerteza do Prophet (margem de erro)
plt.fill_between(teste['ds'], np.clip(previsoes['yhat_lower'], 0, None), previsoes['yhat_upper'], color='dodgerblue', alpha=0.2, label='Margem de Confiança')

plt.title(f"Teste do Prophet em Dados Nunca Vistos (2023-2024): {nome_cidade}", fontsize=14)
plt.ylabel("Casos de Dengue")
plt.legend()
plt.show()

C:\Users\ruiga\AppData\Local\Temp\ipykernel_25488\2455448270.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
